<a href="https://colab.research.google.com/github/remyaP12/labcycle_3sem/blob/main/1labcycle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.Implement a basic Recurrent Neural Network (RNN) to predict the next character in a given sequence. Dataset can be dummy sequential data (e.g., character-level Shakespeare text)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense


In [ ]:
with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()

print("Total characters:", len(text))


Total characters: 1115393


In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

print("Unique characters:", vocab_size)

char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}


Unique characters: 39


In [ ]:
SEQ_LENGTH = 40
step = 3

x_data = []
y_data = []

for i in range(0, len(text) - SEQ_LENGTH, step):
    seq = text[i:i + SEQ_LENGTH]
    target = text[i + SEQ_LENGTH]

    x_data.append([char_to_idx[c] for c in seq])
    y_data.append(char_to_idx[target])

print("Number of sequences:", len(x_data))


Number of sequences: 371785


In [ ]:
x_data = tf.keras.utils.to_categorical(x_data, num_classes=vocab_size)
y_data = np.array(y_data)


In [ ]:
model = Sequential([
    SimpleRNN(128, input_shape=(SEQ_LENGTH, vocab_size)),
    Dense(vocab_size, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy'
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        21,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 39)             │         5,031 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,535 (103.65 KB)

 Trainable params: 26,535 (103.65 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.fit(
    x_data, y_data,
    epochs=10,
    batch_size=128
)


Epoch 1/10
2905/2905 ━━━━━━━━━━━━━━━━━━━━ 90s 31ms/step - loss: 1.6397
Epoch 2/10
2905/2905 ━━━━━━━━━━━━━━━━━━━━ 141s 30ms/step - loss: 1.6247
Epoch 3/10
2905/2905 ━━━━━━━━━━━━━━━━━━━━ 88s 30ms/step - loss: 1.6407
Epoch 4/10
2905/2905 ━━━━━━━━━━━━━━━━━━━━ 92s 32ms/step - loss: 1.6105
Epoch 5/10
2905/2905 ━━━━━━━━━━━━━━━━━━━━ 90s 31ms/step - loss: 1.6059
Epoch 6/10
2905/2905 ━━━━━━━━━━━━━━━━━━━━ 89s 31ms/step - loss: 1.5953
Epoch 7/10
2905/2905 ━━━━━━━━━━━━━━━━━━━━ 89s 31ms/step - loss: 1.5908
Epoch 8/10
2905/2905 ━━━━━━━━━━━━━━━━━━━━ 87s 30ms/step - loss: 1.5792
Epoch 9/10
2905/2905 ━━━━━━━━━━━━━━━━━━━━ 89s 31ms/step - loss: 1.5816
Epoch 10/10
2905/2905 ━━━━━━━━━━━━━━━━━━━━ 92s 32ms/step - loss: 1.5705


In [37]:
def predict_next_char(seed_text, temperature=0.5):
    seed_text = seed_text.lower()
    seed_text = seed_text[-SEQ_LENGTH:].rjust(SEQ_LENGTH)

    seed_encoded = [char_to_idx.get(c, 0) for c in seed_text]
    seed_encoded = tf.keras.utils.to_categorical(
        seed_encoded, num_classes=vocab_size
    )
    seed_encoded = seed_encoded.reshape(1, SEQ_LENGTH, vocab_size)

    preds = model.predict(seed_encoded, verbose=0)[0]
    preds = np.log(preds + 1e-8) / temperature
    preds = np.exp(preds) / np.sum(np.exp(preds))

    return idx_to_char[np.random.choice(len(preds), p=preds)]


In [78]:
seed = "Before we proceed any furthe"
print("Seed text:", seed)
print("Predicted next character:", predict_next_char(seed))


Seed text: Before we proceed any furthe
Predicted next character: r
